In [2]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv('EDA-analysis.csv')
df.info()
df.columns

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 27 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Unnamed: 0       891 non-null    int64  
 1   passengerid      891 non-null    int64  
 2   survived         891 non-null    int64  
 3   pclass           891 non-null    int64  
 4   name             891 non-null    str    
 5   sex              891 non-null    str    
 6   age              714 non-null    float64
 7   sibsp            891 non-null    int64  
 8   parch            891 non-null    int64  
 9   ticket           891 non-null    str    
 10  fare             891 non-null    float64
 11  cabin            204 non-null    str    
 12  embarked         889 non-null    str    
 13  title            891 non-null    str    
 14  age_filled       891 non-null    float64
 15  age_group        891 non-null    str    
 16  family           891 non-null    int64  
 17  family_size      891 non-nu

Index(['Unnamed: 0', 'passengerid', 'survived', 'pclass', 'name', 'sex', 'age',
       'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked', 'title',
       'age_filled', 'age_group', 'family', 'family_size', 'log_fare',
       'class_1', 'embarked_filled', 'avg_fare', 'log_fare2', 'children',
       'fare_new', 'log_fare3', 'sex_numeric'],
      dtype='str')

# Logistic Regression

In [9]:
# feature selection（pclass, sex_numeric, age_group, family_size, log_fare3, embarked_filled)
features = ['pclass','sex_numeric','age_group','family_size','log_fare3','embarked_filled']
df_train = df[features]

# categorical variable encoding
encoded_data = pd.get_dummies(data = df_train,columns=['pclass','age_group','family_size','embarked_filled'],drop_first=True)
encoded_data.head()

# feature scaling
print(encoded_data.log_fare3.describe())
scaler = StandardScaler()
encoded_data['log_fare3'] = scaler.fit_transform(encoded_data[['log_fare3']])
print(encoded_data.log_fare3.describe())
print(encoded_data.columns)

# split into train set and cross-validation set
x = encoded_data
y = df.survived
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=0)
print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

# bulid the model
model1 = LogisticRegression(C=0.09)  # penalty(regularization), class_weight(sample weight), C(regularization parameter), max_iter

# train the model
model1.fit(x_train,y_train)

# observe the coefficient (consistent with EDA or not)
print(model1.coef_)
df_coef = pd.DataFrame(model1.coef_[0], index=x.columns)
print(df_coef)

# compare the fraction of misclassified samples (cross-validation error / train error)
train_error = 1 - model1.score(x_train,y_train)
cv_error = 1 - model1.score(x_test,y_test)
print(f'train error:{train_error * 100:.2f}%')
print(f'cross-validation error:{cv_error * 100:.2f}%')


count    891.000000
mean       2.594874
std        0.773467
min        0.000000
25%        2.169054
50%        2.339881
75%        3.256643
max        5.406181
Name: log_fare3, dtype: float64
count    8.910000e+02
mean     3.508853e-16
std      1.000562e+00
min     -3.356745e+00
25%     -5.508434e-01
50%     -3.298603e-01
75%      8.560689e-01
max      3.636726e+00
Name: log_fare3, dtype: float64
Index(['sex_numeric', 'log_fare3', 'pclass_2', 'pclass_3',
       'age_group_Children', 'age_group_Senior', 'age_group_Teenager',
       'family_size_1', 'family_size_2', 'embarked_filled_Q',
       'embarked_filled_S'],
      dtype='str')
(623, 11) (623,)
(268, 11) (268,)
[[ 1.68637826  0.38023092  0.10782095 -0.59090435  0.58871389 -0.19799904
   0.05927589  0.32192479 -0.36456001  0.0762848  -0.35530276]]
                           0
sex_numeric         1.686378
log_fare3           0.380231
pclass_2            0.107821
pclass_3           -0.590904
age_group_Children  0.588714
age_group_Seni

In [66]:
df_test = pd.read_csv('test.csv')
df_test.columns = df_test.columns.str.lower()
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   passengerid  418 non-null    int64  
 1   pclass       418 non-null    int64  
 2   name         418 non-null    str    
 3   sex          418 non-null    str    
 4   age          332 non-null    float64
 5   sibsp        418 non-null    int64  
 6   parch        418 non-null    int64  
 7   ticket       418 non-null    str    
 8   fare         417 non-null    float64
 9   cabin        91 non-null     str    
 10  embarked     418 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 36.1 KB


In [87]:
# missing value (age:use the same logic in train set or just find the correspnding median age of the same missing titles in train set? )
df.head()

,Unnamed: 0,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,...,family_size,log_fare,class_1,embarked_filled,avg_fare,log_fare2,children,fare_new,log_fare3,sex_numeric
0,0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,...,1,2.110213,0,S,3.62500,1.531476,0,7.2500,2.110213,0
1,1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,...,1,4.280593,1,C,35.64165,3.601186,0,71.2833,4.280593,1
2,2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,...,0,2.188856,0,S,7.92500,2.188856,0,7.9250,2.188856,1
3,3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,...,1,3.990834,1,S,26.55000,3.316003,0,26.5500,3.316003,1
4,4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,...,0,2.202765,0,S,8.05000,2.202765,0,8.0500,2.202765,0
